In [14]:
import os 
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [15]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512,10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

    


In [16]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [17]:
X = torch.rand(1 , 28 , 28, device =device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([7], device='mps:0')


In [18]:
input_image = torch.rand(3, 28, 28)
print(input_image.size())

flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

layer1 = nn.Linear(in_features= 28*28, out_features= 20)
hidden1 = layer1(flat_image)
print(hidden1.size())

print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

torch.Size([3, 28, 28])
torch.Size([3, 784])
torch.Size([3, 20])
Before ReLU: tensor([[ 0.3986, -0.1198, -0.1334,  0.1526, -0.0450,  0.0560,  0.1051,  0.1408,
          0.6693,  0.2774,  0.0133, -0.5213, -0.3186, -0.4747, -0.0129,  0.2544,
          0.1084,  0.8372, -0.0095, -0.1319],
        [ 0.2453,  0.2721, -0.6465,  0.2629,  0.0822, -0.2482,  0.1976, -0.0221,
          0.5089,  0.2296,  0.2796, -0.5159, -0.0461, -0.5354,  0.2541,  0.3521,
          0.1778,  0.1031,  0.3250, -0.2998],
        [ 0.2353,  0.1672, -0.5069,  0.3241, -0.1625,  0.0585,  0.1962,  0.0668,
          0.2992,  0.2206,  0.3097, -0.3236,  0.1261, -0.3204,  0.0484,  0.1672,
         -0.1045,  0.7070,  0.4488, -0.1224]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.3986, 0.0000, 0.0000, 0.1526, 0.0000, 0.0560, 0.1051, 0.1408, 0.6693,
         0.2774, 0.0133, 0.0000, 0.0000, 0.0000, 0.0000, 0.2544, 0.1084, 0.8372,
         0.0000, 0.0000],
        [0.2453, 0.2721, 0.0000, 0.2629, 0.0822, 0.0000, 0.1976, 0.00

In [22]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20,10)
)
input_image = torch.rand(3, 28,28)
logits = seq_modules(input_image)

softmax = nn.Softmax(dim = 1)
pred_probab = softmax(logits)

In [23]:
print(f"Model structure: {model}\n\n")

for name , param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values: {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values: tensor([[-0.0155,  0.0172, -0.0353,  ..., -0.0157, -0.0307, -0.0067],
        [-0.0027, -0.0353,  0.0246,  ..., -0.0058, -0.0102,  0.0249]],
       device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values: tensor([-0.0064,  0.0230], device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values: tensor([[ 0.0033, -0.0064,  0.0406,  ...,  0.0414, -0.0025, -0.0167],
        [-0.0306, -0.0172, -0.0139,  ...,  0.0200,  0.0007,  0.0076]],
       device='mps:0', grad_fn=<SliceBac